# GLM Encoding Model Fitting

This notebook fits a Generalized Linear Model (GLM) to predict neural activity from task predictors.

**Training Strategy:**
- Train on Context 1 & 2 data (audio and visual contexts)
- Test on Context 0 data (congruent context)

**Model:**
- Poisson GLM with exponential activation
- Group Lasso regularization
- 5-fold cross-validation for model selection

## 1. Imports and Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import scipy.stats
import os
from matplotlib import rcParams
import sys

# Import GLM classes
sys.path.insert(0, '/Users/akhilbandi/Documents/Portfolio/code_projects/Context Paper Code/GLM_encoding_model')
import glm_class

# Set plotting style
rcParams['font.family'] = 'Arial'
rcParams['axes.spines.right'] = False
rcParams['axes.spines.top'] = False

print("✓ Imports complete")

✓ Imports complete


## 2. Load Pre-processed Data

The data was prepared in `Create_task_predictors.ipynb` with:
- **X**: Design matrix (z-scored along sample dimension)
- **Y**: Neural activity (deconv × 10 to mimic spike counts)
- **metadata**: Frame-level trial information for splitting

## 2. Load Pre-processed Data

The data was prepared in `Create_task_predictors.ipynb` with:
- **X**: Design matrix (z-scored along sample dimension)
- **Y**: Neural activity (deconv × 10 to mimic spike counts)
- **metadata**: Frame-level trial information for splitting

In [ ]:
# Define dataset
mouse_ID = 'KA-2-00'
date = '2026-01-28'
load_path = f'/Volumes/Akhil Data/Akhil/ProcessedData/{mouse_ID}/{date}/glm_data_stacked.pkl'

print(f"Loading data from: {load_path}")

with open(load_path, 'rb') as f:
    glm_data = pickle.load(f)

# Extract components
X = glm_data['X']  # Already z-scored
Y = glm_data['Y']  # Already scaled by 10 for deconv
metadata = glm_data['metadata']
predictor_names = glm_data['predictor_names']
predictor_info = glm_data['predictor_info']
group_indices = glm_data['group_indices']

print(f"\n✓ Data loaded successfully!")
print(f"  X shape: {X.shape} (frames × predictors)")
print(f"  Y shape: {Y.shape} (frames × neurons)")
print(f"  Metadata keys: {list(metadata.keys())}")
print(f"  Predictor groups: {list(predictor_info.keys())}")

# Check if constant predictor exists and remove it (for old data files)
if 'constant' in predictor_info:
    print("⚠️  Detected old data format with constant predictor - removing it...")
    
    const_start = predictor_info['constant']['start']
    const_end = predictor_info['constant']['end']
    
    # Remove constant column from X
    keep_mask = np.ones(X.shape[1], dtype=bool)
    keep_mask[const_start:const_end] = False
    X = X[:, keep_mask]
    
    # Update predictor_info
    predictor_info_clean = {}
    offset = 0
    for group_name, info in predictor_info.items():
        if group_name == 'constant':
            offset = const_end - const_start
            continue
        predictor_info_clean[group_name] = {
            'start': info['start'] if info['start'] < const_start else info['start'] - offset,
            'end': info['end'] if info['end'] < const_start else info['end'] - offset,
            'names': info['names']
        }
    predictor_info = predictor_info_clean
    
    # Update group_indices
    group_indices_clean = []
    for i, indices in enumerate(group_indices):
        if isinstance(indices, (list, np.ndarray)):
            # Keep only indices that are not in the constant range
            clean_idx = [idx for idx in indices if idx < const_start or idx >= const_end]
            # Adjust indices
            clean_idx = [idx if idx < const_start else idx - offset for idx in clean_idx]
            if len(clean_idx) > 0:
                group_indices_clean.append(clean_idx)
        else:
            # If group_indices is list of names, skip 'constant'
            if indices != 'constant':
                group_indices_clean.append(indices)
    group_indices = group_indices_clean
    
    # Update predictor_names if it exists
    if predictor_names is not None:
        predictor_names = [predictor_names[i] for i in range(len(predictor_names)) if keep_mask[i]]
    
    print(f"✓ Removed constant predictor")
    print(f"  New X shape: {X.shape}")
    print(f"  Predictor groups: {list(predictor_info.keys())}")
else:
    print("✓ Data already clean (no constant predictor)")

print(f"\nFinal X shape: {X.shape}")

In [2]:
# This cell is now merged with the previous cell - delete this one

Loading data from: /Volumes/Akhil Data/Akhil/ProcessedData/KA-2-00/2026-01-28/glm_data_stacked.pkl

✓ Data loaded successfully!
  X shape: (19795, 146) (frames × predictors)
  Y shape: (19795, 2122) (frames × neurons)
  Metadata keys: ['trial_id', 'context', 'choice', 'outcome', 'audio_stim', 'visual_stim', 'trial_type', 'frame_in_trial']
  Predictor groups: ['position_only', 'sound', 'visual', 'velocity', 'choice_spatial', 'turn_onset', 'reward', 'constant']


### Verify Data Preprocessing

Check that X and Y are properly formatted per Harvey Lab tutorial:
- X should be z-scored (mean≈0, std≈1)
- Y should be scaled by 10× for deconv
- No NaN or Inf values

In [3]:
print("Data Verification")
print("="*60)

# Check X (design matrix)
print("\nX (Design Matrix):")
print(f"  Mean: {np.mean(X):.6f} (should be ≈0)")
print(f"  Std:  {np.std(X):.3f} (should be ≈1)")
print(f"  Range: [{np.min(X):.2f}, {np.max(X):.2f}]")
print(f"  NaN count: {np.sum(np.isnan(X))}")
print(f"  Inf count: {np.sum(np.isinf(X))}")

# Check Y (neural data)
print("\nY (Neural Data):")
print(f"  Mean: {np.mean(Y):.3f}")
print(f"  Std:  {np.std(Y):.3f}")
print(f"  Range: [{np.min(Y):.2f}, {np.max(Y):.2f}]")
print(f"  NaN count: {np.sum(np.isnan(Y))}")
print(f"  Inf count: {np.sum(np.isinf(Y))}")

# Check metadata
print("\nMetadata (Context distribution):")
for ctx in [0, 1, 2]:
    n_frames = np.sum(metadata['context'] == ctx)
    pct = n_frames / len(metadata['context']) * 100
    print(f"  Context {ctx}: {n_frames:,} frames ({pct:.1f}%)")

print("\n✓ Data verification complete")

Data Verification

X (Design Matrix):
  Mean: -0.000000 (should be ≈0)
  Std:  0.998 (should be ≈1)
  Range: [-1.29, 130.37]
  NaN count: 0
  Inf count: 0

Y (Neural Data):
  Mean: 0.017
  Std:  0.171
  Range: [0.00, 18.37]
  NaN count: 0
  Inf count: 0

Metadata (Context distribution):
  Context 0: 6,658 frames (33.6%)
  Context 1: 8,170 frames (41.3%)
  Context 2: 4,967 frames (25.1%)

✓ Data verification complete


## 3. Train/Test Split by Context

**Training:** Context 1 (audio) + Context 2 (visual)  
**Testing:** Context 0 (congruent)

In [4]:
# Create masks for train and test sets
train_mask = (metadata['context'] == 1) | (metadata['context'] == 2)
test_mask = metadata['context'] == 0

# Split data
X_train = X[train_mask]
Y_train = Y[train_mask]
trial_id_train = metadata['trial_id'][train_mask]

X_test = X[test_mask]
Y_test = Y[test_mask]
trial_id_test = metadata['trial_id'][test_mask]

print("Train/Test Split by Context")
print("="*60)
print(f"\nTraining Set (Context 1 & 2):")
print(f"  Frames: {X_train.shape[0]:,} ({X_train.shape[0]/X.shape[0]*100:.1f}%)")
print(f"  Trials: {len(np.unique(trial_id_train))}")

print(f"\nTest Set (Context 0):")
print(f"  Frames: {X_test.shape[0]:,} ({X_test.shape[0]/X.shape[0]*100:.1f}%)")
print(f"  Trials: {len(np.unique(trial_id_test))}")

# Show balance within training set
print(f"\nTraining Set Balance:")
for ctx in [1, 2]:
    ctx_mask = metadata['context'][train_mask] == ctx
    n_frames = np.sum(ctx_mask)
    pct = n_frames / len(train_mask) * 100
    print(f"  Context {ctx}: {n_frames:,} frames ({pct:.1f}%)")

print("\n✓ Data split complete")

Train/Test Split by Context

Training Set (Context 1 & 2):
  Frames: 13,137 (66.4%)
  Trials: 60

Test Set (Context 0):
  Frames: 6,658 (33.6%)
  Trials: 29

Training Set Balance:
  Context 1: 8,170 frames (41.3%)
  Context 2: 4,967 frames (25.1%)

✓ Data split complete


### Prepare Feature Groups for Group Lasso

In [5]:
def get_feature_group_size(group_indices):
    """
    Convert group_indices to feature_group_size for GLM_CV.
    
    group_indices: list of arrays, each containing column indices for one group
    Returns: array of group sizes
    """
    return np.array([len(g) for g in group_indices])

feature_group_size = get_feature_group_size(group_indices)

print("Feature Groups for Group Lasso:")
print("="*60)
for i, (group_name, group_data) in enumerate(predictor_info.items()):
    print(f"{group_name:20s}: {feature_group_size[i]:2d} features")
print(f"\nTotal: {np.sum(feature_group_size)} features")
print("\n✓ Feature groups ready")

Feature Groups for Group Lasso:
position_only       : 13 features
sound               : 13 features
visual              : 13 features
velocity            : 13 features
choice_spatial      : 13 features
turn_onset          : 13 features
reward              : 13 features
constant            : 13 features

Total: 1478 features

✓ Feature groups ready


### Check GPU Availability

In [6]:
import tensorflow as tf
import multiprocessing

print("System Resources:")
print("="*60)
print(f"TensorFlow version: {tf.__version__}")

# Check for Metal GPU on Mac
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✓ GPU Available: {len(gpus)} device(s)")
    for gpu in gpus:
        print(f"  - {gpu}")
    
    # Enable memory growth to avoid allocating all GPU memory at once
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("  ✓ Memory growth enabled")
    except:
        pass
else:
    print("\n⚠️  No GPU detected")
    print("  For Mac with Apple Silicon, install tensorflow-metal:")
    print("  pip install tensorflow-metal")

# CPU info
n_cores = multiprocessing.cpu_count()
print(f"\nCPU cores: {n_cores}")
print("="*60)

System Resources:
TensorFlow version: 2.16.2

✓ GPU Available: 1 device(s)
  - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
  ✓ Memory growth enabled

CPU cores: 24


## 4. Fit GLM with Cross-Validation

Using Harvey Lab tutorial settings:
- Poisson GLM with exp activation
- Group Lasso regularization
- 5-fold cross-validation on training data
- Lambda series from 10^-1 to 10^-6

In [7]:
# Clear any previous keras states
tf.keras.backend.clear_session()

# Initialize GLM_CV (following Harvey Lab tutorial)
model_cv = glm_class.GLM_CV(
    n_folds=5,
    auto_split=True,
    split_by_group=True,
    activation='exp',
    loss_type='poisson',
    regularization='group_lasso',
    lambda_series=10.0 ** np.linspace(-1, -6, 11),
    l1_ratio=0.0,
    smooth_strength=0.1,
    optimizer='adam',
    learning_rate=1e-2,
    convergence_tol=1e-6
)

print("Model Configuration:")
print("="*60)
print(f"  Activation: {model_cv.activation}")
print(f"  Loss: {model_cv.loss_type}")
print(f"  Regularization: {model_cv.regularization}")
print(f"  Lambda range: [{model_cv.lambda_series.min():.2e}, {model_cv.lambda_series.max():.2e}]")
print(f"  N lambdas: {len(model_cv.lambda_series)}")
print(f"  CV folds: {model_cv.n_folds}")
print(f"  Learning rate: {model_cv.learning_rate}")
print("="*60)
print("\n✓ Model initialized")

Model Configuration:
  Activation: exp
  Loss: poisson
  Regularization: group_lasso
  Lambda range: [1.00e-06, 1.00e-01]
  N lambdas: 11
  CV folds: 5
  Learning rate: 0.01

✓ Model initialized


In [8]:
print("Fitting GLM with 5-fold cross-validation...\n")
print(f"Training on {Y_train.shape[1]} neurons")
print(f"Training frames: {X_train.shape[0]:,}")
print(f"Predictors: {X_train.shape[1]}")
print("="*60)

# Fit the model
model_cv.fit(
    X_train,
    Y_train,
    group_idx=trial_id_train,
    feature_group_size=feature_group_size,
    verbose=True
)

print("\n✓ Model fitting complete!")

Fitting GLM with 5-fold cross-validation...

Training on 2122 neurons
Training frames: 13,137
Predictors: 146


2026-02-10 08:42:22.322525: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Ultra
2026-02-10 08:42:22.322558: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-02-10 08:42:22.322564: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 53.76 GB
2026-02-10 08:42:22.322583: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-02-10 08:42:22.322595: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


AssertionError: Error: Sum of group_size_list is not equal to number of features (X.shape[1])!

## 5. Model Selection

Select optimal lambda per neuron using 1-SE rule on CV performance

In [ ]:
# Select best model for each neuron
model_cv.select_model(make_fig=True)

print("\nModel Selection Results:")
print("="*60)
print(f"Mean CV frac_dev: {np.mean(model_cv.selected_frac_dev_expl_cv):.3f}")
print(f"Median CV frac_dev: {np.median(model_cv.selected_frac_dev_expl_cv):.3f}")
print(f"Range: [{np.min(model_cv.selected_frac_dev_expl_cv):.3f}, {np.max(model_cv.selected_frac_dev_expl_cv):.3f}]")

print(f"\nSelected lambda range: [{model_cv.selected_lambda.min():.2e}, {model_cv.selected_lambda.max():.2e}]")
print(f"Median selected lambda: {np.median(model_cv.selected_lambda):.2e}")

# Count well-modeled neurons
good_neurons = np.sum(model_cv.selected_frac_dev_expl_cv > 0.1)
print(f"\nNeurons with frac_dev > 0.1: {good_neurons}/{Y_train.shape[1]} ({good_neurons/Y_train.shape[1]*100:.1f}%)")
print("="*60)

## 6. Evaluate on Test Set (Context 0)

Test generalization to held-out congruent context data

In [ ]:
# Make predictions on test set
from glm_class import make_prediction, deviance

Y_test_pred = make_prediction(
    model_cv.selected_w,
    X_test,
    activation=model_cv.activation
)

# Calculate deviance explained on test set
test_frac_dev = np.zeros(Y_test.shape[1])
for i in range(Y_test.shape[1]):
    dev_test = deviance(
        Y_test[:, i],
        Y_test_pred[:, i],
        loss_type=model_cv.loss_type
    )
    dev_null = deviance(
        Y_test[:, i],
        np.mean(Y_test[:, i]),
        loss_type=model_cv.loss_type
    )
    test_frac_dev[i] = 1 - (dev_test / dev_null)

print("Test Set Performance (Context 0):")
print("="*60)
print(f"Mean frac_dev: {np.mean(test_frac_dev):.3f}")
print(f"Median frac_dev: {np.median(test_frac_dev):.3f}")
print(f"Range: [{np.min(test_frac_dev):.3f}, {np.max(test_frac_dev):.3f}]")

good_test = np.sum(test_frac_dev > 0.1)
print(f"\nNeurons with frac_dev > 0.1: {good_test}/{Y_test.shape[1]} ({good_test/Y_test.shape[1]*100:.1f}%)")
print("="*60)

## 7. Visualize Performance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. CV performance histogram
ax = axes[0, 0]
ax.hist(model_cv.selected_frac_dev_expl_cv, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
ax.axvline(np.median(model_cv.selected_frac_dev_expl_cv), color='red', linestyle='--', label='Median')
ax.axvline(0.1, color='green', linestyle='--', label='0.1 threshold')
ax.set_xlabel('Fraction Deviance Explained (CV)')
ax.set_ylabel('Number of Neurons')
ax.set_title('Cross-Validation Performance')
ax.legend()

# 2. Test performance histogram
ax = axes[0, 1]
ax.hist(test_frac_dev, bins=50, alpha=0.7, color='coral', edgecolor='black')
ax.axvline(np.median(test_frac_dev), color='red', linestyle='--', label='Median')
ax.axvline(0.1, color='green', linestyle='--', label='0.1 threshold')
ax.set_xlabel('Fraction Deviance Explained (Test)')
ax.set_ylabel('Number of Neurons')
ax.set_title('Test Set Performance (Context 0)')
ax.legend()

# 3. CV vs Test scatter
ax = axes[1, 0]
ax.scatter(model_cv.selected_frac_dev_expl_cv, test_frac_dev, alpha=0.3, s=10)
ax.plot([-1, 1], [-1, 1], 'k--', alpha=0.5, label='Unity')
ax.set_xlabel('CV Performance')
ax.set_ylabel('Test Performance')
ax.set_title('CV vs Test Performance')
ax.legend()
ax.set_xlim(-0.5, max(model_cv.selected_frac_dev_expl_cv.max(), test_frac_dev.max()) + 0.1)
ax.set_ylim(-0.5, max(model_cv.selected_frac_dev_expl_cv.max(), test_frac_dev.max()) + 0.1)

# 4. Cumulative distribution
ax = axes[1, 1]
sorted_cv = np.sort(model_cv.selected_frac_dev_expl_cv)
sorted_test = np.sort(test_frac_dev)
ax.plot(sorted_cv, np.arange(len(sorted_cv)) / len(sorted_cv), label='CV', linewidth=2)
ax.plot(sorted_test, np.arange(len(sorted_test)) / len(sorted_test), label='Test', linewidth=2)
ax.axvline(0.1, color='green', linestyle='--', alpha=0.5, label='0.1 threshold')
ax.set_xlabel('Fraction Deviance Explained')
ax.set_ylabel('Cumulative Fraction of Neurons')
ax.set_title('Cumulative Distribution')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete")

## 8. Save Results

In [ ]:
# Prepare results dictionary
results = {
    # Model weights and parameters
    'weights': model_cv.selected_w,
    'selected_lambda': model_cv.selected_lambda,
    
    # Performance metrics
    'cv_frac_dev': model_cv.selected_frac_dev_expl_cv,
    'test_frac_dev': test_frac_dev,
    
    # Metadata
    'predictor_names': predictor_names,
    'predictor_info': predictor_info,
    'feature_group_size': feature_group_size,
    
    # Model configuration
    'model_config': {
        'activation': model_cv.activation,
        'loss_type': model_cv.loss_type,
        'regularization': model_cv.regularization,
        'lambda_series': model_cv.lambda_series,
        'n_folds': model_cv.n_folds,
        'learning_rate': model_cv.learning_rate,
    },
    
    # Dataset info
    'mouse_ID': mouse_ID,
    'date': date,
    'train_contexts': [1, 2],
    'test_context': 0,
}

# Save results
save_dir = f'/Volumes/Akhil Data/Akhil/ProcessedData/{mouse_ID}/{date}/'
save_path = os.path.join(save_dir, 'glm_results.pkl')

with open(save_path, 'wb') as f:
    pickle.dump(results, f)

print(f"Results saved to: {save_path}")
print(f"File size: {os.path.getsize(save_path) / 1e6:.2f} MB")
print("\n✓ Save complete")

## Summary

This notebook:
1. Loaded pre-processed GLM data (X, Y, metadata)
2. Split into train (Context 1&2) and test (Context 0) sets
3. Fitted Poisson GLM with Group Lasso regularization
4. Selected optimal models using 5-fold cross-validation
5. Evaluated generalization on held-out test data
6. Visualized performance distributions
7. Saved results for downstream analysis